# OMIP Simulation Diagnostics

Post-processing visualization loosely following Adcroft et al. (2019),
*The GFDL Global Ocean and Sea Ice Model OM4.0*, JAMES.

1. Time-mean SST / SSS and bias vs WOA
2. SSH, MLD, sea-ice concentration (March & September)
3. Surface heat and freshwater fluxes
4. Global-mean T & S drift, horizontal-mean profiles
5. Zonal-mean T, S, b and difference from initial conditions (WOA),
   computed via `ConservativeRegridding` to a regular lat-lon grid

In [ ]:
run_dir = "halfdegree_run"   # <-- path to the _run folder
prefix  = replace(basename(run_dir), "_run" => "")

In [ ]:
using CairoMakie
using Statistics
using Dates
using Oceananigans
using Oceananigans.Grids: znodes, φnodes
using Oceananigans.Fields: interpolate!
using ConservativeRegridding
using NumericalEarth
using NumericalEarth.DataWrangling: Metadatum
using NumericalEarth.DataWrangling.WOA: WOAAnnual

In [ ]:
# ── Helpers ──────────────────────────────────────────────────

function find_first_file(run_dir, prefix, group)
    tag = "$(prefix)_$(group)"

    files = filter(f ->
        startswith(f, tag) &&
        endswith(f, ".jld2") &&
        !contains(f, "checkpoint"),
        readdir(run_dir)
    )

    isempty(files) && error("No $group files found for prefix '$prefix' in $run_dir")

    f = first(sort(files))
    return joinpath(run_dir, replace(f, r"_part\d+\.jld2$" => ".jld2"))
end

function compute_time_mean(fts)
    Nt  = length(fts.times)
    avg = zeros(size(Array(interior(fts[1]))))
    for n in 1:Nt
        avg .+= Array(interior(fts[n]))
    end
    return avg ./ Nt
end

function compute_monthly_mean(fts, target_months; start_date = DateTime(1958, 1, 1))
    dates = [start_date + Second(round(Int, t)) for t in fts.times]
    idx   = findall(d -> month(d) in target_months, dates)
    isempty(idx) && return nothing
    avg = zeros(size(Array(interior(fts[1]))))
    for n in idx
        avg .+= Array(interior(fts[n]))
    end
    return avg ./ length(idx)
end

function build_land_mask(grid)
    if grid isa ImmersedBoundaryGrid
        bh = Array(interior(grid.immersed_boundary.bottom_height, :, :, 1))
        return bh .>= 0
    else
        return falses(size(grid, 1), size(grid, 2))
    end
end

function build_ocean_mask_3d(grid)
    Nx, Ny, Nz = size(grid)
    mask = ones(Nx, Ny, Nz)
    if grid isa ImmersedBoundaryGrid
        bh = Array(interior(grid.immersed_boundary.bottom_height, :, :, 1))
        zc = znodes(grid, Center())
        for k in 1:Nz, j in 1:Ny, i in 1:Nx
            zc[k] < bh[i, j] && (mask[i, j, k] = 0.0)
        end
    end
    return mask
end

mask_land!(f, land) = (f[land] .= NaN; f)

function panel!(fig, pos, data;
                title="", colormap=:thermal,
                colorrange=nothing, label="",
                nan_color=:lightgray)
    ax = Axis(fig[pos...]; title)
    kw = isnothing(colorrange) ? (;) : (; colorrange)
    hm = heatmap!(ax, data; colormap, nan_color, kw...)
    Colorbar(fig[pos[1], pos[2]+1], hm; label)
    return ax
end

function sidebyside!(fig, row, left, right;
                     title_l="", title_r="",
                     cmap_l=:thermal, cmap_r=:balance,
                     cr_l=nothing, cr_r=nothing,
                     label_l="", label_r="")
    panel!(fig, [row, 1], left;  title=title_l, colormap=cmap_l, colorrange=cr_l, label=label_l)
    panel!(fig, [row, 3], right; title=title_r, colormap=cmap_r, colorrange=cr_r, label=label_r)
end

## Load surface diagnostics

In [ ]:
surface_file = find_first_file(run_dir, prefix, "surface")
@info "Surface file: $surface_file"

tos  = FieldTimeSeries(surface_file, "tos";    backend = OnDisk())
sos  = FieldTimeSeries(surface_file, "sos";    backend = OnDisk())
zos  = FieldTimeSeries(surface_file, "zos";    backend = OnDisk())
mld_fts = FieldTimeSeries(surface_file, "mlotst"; backend = OnDisk())
hfds = FieldTimeSeries(surface_file, "hfds";   backend = OnDisk())
wfo  = FieldTimeSeries(surface_file, "wfo";    backend = OnDisk())
sic  = FieldTimeSeries(surface_file, "siconc"; backend = OnDisk())

grid = tos.grid
Nx, Ny, Nz = size(grid)
land = build_land_mask(grid)
@info "Grid: $Nx x $Ny x $Nz   |  $(length(tos.times)) surface snapshots"

In [ ]:
SST = dropdims(compute_time_mean(tos);  dims=3)
SSS = dropdims(compute_time_mean(sos);  dims=3)
SSH = dropdims(compute_time_mean(zos);  dims=3)
MLD_avg = dropdims(compute_time_mean(mld_fts); dims=3)
HF  = dropdims(compute_time_mean(hfds); dims=3)
FW  = dropdims(compute_time_mean(wfo);  dims=3)
SIC = dropdims(compute_time_mean(sic);  dims=3)

## WOA comparison

In [ ]:
T_woa = Field(Metadatum(:temperature; dataset = WOAAnnual()), CPU())
S_woa = Field(Metadatum(:salinity;    dataset = WOAAnnual()), CPU())

T_interp = CenterField(grid)
S_interp = CenterField(grid)
interpolate!(T_interp, T_woa)
interpolate!(S_interp, S_woa)

# Full 3-D WOA on model grid (reused later for zonal-mean bias)
T_woa_on_grid = Array(interior(T_interp))
S_woa_on_grid = Array(interior(S_interp))

SST_woa = T_woa_on_grid[:, :, Nz]
SSS_woa = S_woa_on_grid[:, :, Nz]

δSST = SST .- SST_woa
δSSS = SSS .- SSS_woa

for f in (SST, SSS, SSH, MLD_avg, HF, FW, SIC, δSST, δSSS)
    mask_land!(f, land)
end

## Figure 1 -- SST and WOA bias (cf. OM4 Fig. 3)

In [ ]:
fig = Figure(size = (1600, 550), fontsize = 14)
sidebyside!(fig, 1, SST, δSST;
            title_l = "Time-mean SST",  title_r = "SST - WOA",
            cmap_l = :thermal, cr_l = (-2, 32),
            cmap_r = :balance, cr_r = (-5, 5),
            label_l = "deg C", label_r = "deg C")
fig

## Figure 2 -- SSS and WOA bias (cf. OM4 Fig. 4)

In [ ]:
fig = Figure(size = (1600, 550), fontsize = 14)
sidebyside!(fig, 1, SSS, δSSS;
            title_l = "Time-mean SSS", title_r = "SSS - WOA",
            cmap_l = :haline, cr_l = (30, 38),
            cmap_r = :balance, cr_r = (-3, 3),
            label_l = "PSU", label_r = "PSU")
fig

## Figure 3 -- SSH (cf. OM4 Fig. 5)

In [ ]:
fig = Figure(size = (900, 500), fontsize = 14)
panel!(fig, [1, 1], SSH;
       title = "Time-mean SSH", colormap = :balance,
       colorrange = (-2, 2), label = "m")
fig

## Figure 4 -- Min / Max monthly-mean MLD (cf. OM4 Figs. 3-4)

In [ ]:
# For each grid point, find the min and max MLD across all 12 monthly means
MLD_monthly = [compute_monthly_mean(mld_fts, [m]) for m in 1:12]
available = findall(!isnothing, MLD_monthly)
MLD_stack = cat([dropdims(MLD_monthly[m]; dims=3) for m in available]...; dims=3)

MLD_min = dropdims(minimum(MLD_stack; dims=3); dims=3)
MLD_max = dropdims(maximum(MLD_stack; dims=3); dims=3)
mask_land!(MLD_min, land)
mask_land!(MLD_max, land)

fig = Figure(size = (1600, 550), fontsize = 14)
panel!(fig, [1, 1], MLD_min; title="Minimum monthly-mean MLD (summer)",
       colormap=Reverse(:deep), colorrange=(0, 150), label="m")
panel!(fig, [1, 3], MLD_max; title="Maximum monthly-mean MLD (winter)",
       colormap=Reverse(:deep), colorrange=(10, 3000), label="m")
fig

## Figure 5 -- Sea-ice concentration March / September (cf. OM4 Figs. 9-10)

In [ ]:
SIC_mar = compute_monthly_mean(sic, [3])
SIC_sep = compute_monthly_mean(sic, [9])

fig = Figure(size = (1600, 550), fontsize = 14)
if !isnothing(SIC_mar)
    d = dropdims(SIC_mar; dims=3); mask_land!(d, land)
    panel!(fig, [1, 1], d; title="Sea-ice conc. -- March",
           colormap=:ice, colorrange=(0, 1), label="fraction")
end
if !isnothing(SIC_sep)
    d = dropdims(SIC_sep; dims=3); mask_land!(d, land)
    panel!(fig, [1, 3], d; title="Sea-ice conc. -- September",
           colormap=:ice, colorrange=(0, 1), label="fraction")
end
fig

## Figure 6 -- Surface fluxes (cf. OM4 Figs. 11-12)

In [ ]:
fig = Figure(size = (1600, 550), fontsize = 14)
sidebyside!(fig, 1, HF, FW;
            title_l = "Net surface heat flux",
            title_r = "Net freshwater flux",
            cmap_l = :balance, cr_l = (-200, 200),
            cmap_r = :balance, cr_r = (-1e-4, 1e-4),
            label_l = "W/m^2", label_r = "kg/m^2/s")
fig

## Figure 7 -- SSH variance

In [ ]:
# SSH variance: Var(η) = <η²> - <η>²
zossq_fts = FieldTimeSeries(surface_file, "zossq"; backend = OnDisk())
SSH_sq_mean = dropdims(compute_time_mean(zossq_fts); dims=3)
SSH_var = SSH_sq_mean .- SSH .^ 2
mask_land!(SSH_var, land)

fig = Figure(size = (900, 500), fontsize = 14)
panel!(fig, [1, 1], SSH_var;
       title = "SSH variance", colormap = :magma,
       colorrange = (0, 0.05), label = "m²")
fig

## Figure 8 -- Global-mean TKE

In [ ]:
# Volume-mean TKE time series from 3-D CATKE output
fields_file = find_first_file(run_dir, prefix, "fields")
tke_fts = FieldTimeSeries(fields_file, "tke"; backend = OnDisk())

ocean_mask = build_ocean_mask_3d(grid)
N_ocean = sum(ocean_mask)

tke_global = Float64[]
for n in 1:length(tke_fts.times)
    e = Array(interior(tke_fts[n]))
    push!(tke_global, sum(e .* ocean_mask) / N_ocean)
end

t_years = tke_fts.times ./ (365.25 * 24 * 3600)

fig = Figure(size = (900, 450), fontsize = 14)
ax = Axis(fig[1, 1]; xlabel = "Time (years)", ylabel = "TKE (m²/s²)",
          title = "Global-mean turbulent kinetic energy")
lines!(ax, t_years, tke_global; color = :black)
fig

## Figure 9 -- Global-mean T and S drift (cf. OM4 Fig. 13)

In [ ]:
avg_file = find_first_file(run_dir, prefix, "averages")

tosga_fts = FieldTimeSeries(avg_file, "tosga"; backend = OnDisk())
soga_fts  = FieldTimeSeries(avg_file, "soga";  backend = OnDisk())

tosga = [Array(interior(tosga_fts[n]))[1] for n in 1:length(tosga_fts.times)]
soga  = [Array(interior(soga_fts[n]))[1]  for n in 1:length(soga_fts.times)]
t_years = tosga_fts.times ./ (365.25 * 24 * 3600)

fig = Figure(size = (1200, 450), fontsize = 14)
ax = Axis(fig[1, 1]; xlabel="Time (years)", ylabel="dT (deg C)",
          title="Global-mean temperature drift")
lines!(ax, t_years, tosga .- tosga[1]; color=:firebrick)

ax = Axis(fig[1, 2]; xlabel="Time (years)", ylabel="dS (PSU)",
          title="Global-mean salinity drift")
lines!(ax, t_years, soga .- soga[1]; color=:royalblue)
fig

## Figure 10 -- Horizontal-mean T and S profiles (cf. OM4 Fig. 14)

In [ ]:
to_h_fts = FieldTimeSeries(avg_file, "to_h"; backend = OnDisk())
so_h_fts = FieldTimeSeries(avg_file, "so_h"; backend = OnDisk())

T_prof = vec(compute_time_mean(to_h_fts))
S_prof = vec(compute_time_mean(so_h_fts))
z = collect(znodes(grid, Center()))

fig = Figure(size = (1000, 600), fontsize = 14)
ax = Axis(fig[1, 1]; xlabel="Temperature (deg C)", ylabel="Depth (m)",
          title="Horizontal-mean temperature")
lines!(ax, T_prof, z; color=:firebrick)
ylims!(ax, (-5500, 0))

ax = Axis(fig[1, 2]; xlabel="Salinity (PSU)", ylabel="Depth (m)",
          title="Horizontal-mean salinity")
lines!(ax, S_prof, z; color=:royalblue)
ylims!(ax, (-5500, 0))
fig

---
## Zonal-mean sections

Regrid the 3-D time-mean fields from the native (tripolar / ORCA) grid
to a regular 1-degree latitude-longitude grid via `ConservativeRegridding`,
then average over longitude to obtain latitude-depth sections.
An ocean mask is carried through the regridding so that immersed cells
do not contaminate the zonal averages.

In [ ]:
# Target lat-lon grid (1 degree)
Nlon, Nlat = 360, 180
latlon_grid = LatitudeLongitudeGrid(CPU();
    size = (Nlon, Nlat, 1),
    longitude = (0, 360),
    latitude  = (-90, 90),
    z = (0, 1))

src_f = Field{Center, Center, Nothing}(grid)
dst_f = Field{Center, Center, Nothing}(latlon_grid)

@info "Building conservative regridder (this may take a few minutes)..."
regridder = ConservativeRegridding.Regridder(dst_f, src_f; progress = true)
@info "Regridder ready."

In [ ]:
"""Regrid a 3-D field level-by-level and compute the
ocean-area-weighted zonal mean using a carried ocean mask."""
function compute_zonal_mean(data_3d, ocean_mask_3d, regridder, Nlon, Nlat)
    Nz = size(data_3d, 3)
    zonal    = fill(NaN, Nlat, Nz)
    dst_data = zeros(Nlon * Nlat)
    dst_mask = zeros(Nlon * Nlat)
    areas    = regridder.dst_areas

    for k in 1:Nz
        ConservativeRegridding.regrid!(dst_data, regridder,
            vec(data_3d[:, :, k] .* ocean_mask_3d[:, :, k]))
        ConservativeRegridding.regrid!(dst_mask, regridder,
            vec(ocean_mask_3d[:, :, k]))

        data_sum = reshape(dst_data .* areas, Nlon, Nlat)
        mask_sum = reshape(dst_mask .* areas, Nlon, Nlat)

        for j in 1:Nlat
            m = sum(@view mask_sum[:, j])
            m > 0 && (zonal[j, k] = sum(@view data_sum[:, j]) / m)
        end
    end
    return zonal
end

In [ ]:
@info "Loading 3-D field time series..."
fields_file = find_first_file(run_dir, prefix, "fields")

to_fts = FieldTimeSeries(fields_file, "to"; backend = OnDisk())
so_fts = FieldTimeSeries(fields_file, "so"; backend = OnDisk())
bo_fts = FieldTimeSeries(fields_file, "bo"; backend = OnDisk())

@info "$(length(to_fts.times)) field snapshots -- computing time means..."
T_mean = compute_time_mean(to_fts)
S_mean = compute_time_mean(so_fts)
b_mean = compute_time_mean(bo_fts)

# Initial-condition buoyancy (first averaged snapshot)
b_init = Array(interior(bo_fts[1]))

In [ ]:
ocean_mask = build_ocean_mask_3d(grid)

@info "Computing zonal means (model)..."
T_zonal = compute_zonal_mean(T_mean, ocean_mask, regridder, Nlon, Nlat)
S_zonal = compute_zonal_mean(S_mean, ocean_mask, regridder, Nlon, Nlat)
b_zonal = compute_zonal_mean(b_mean, ocean_mask, regridder, Nlon, Nlat)

@info "Computing zonal means (WOA / initial conditions)..."
T_woa_zonal  = compute_zonal_mean(T_woa_on_grid, ocean_mask, regridder, Nlon, Nlat)
S_woa_zonal  = compute_zonal_mean(S_woa_on_grid, ocean_mask, regridder, Nlon, Nlat)
b_init_zonal = compute_zonal_mean(b_init,        ocean_mask, regridder, Nlon, Nlat)

# Differences from initial conditions
δT_zonal = T_zonal .- T_woa_zonal
δS_zonal = S_zonal .- S_woa_zonal
δb_zonal = b_zonal .- b_init_zonal

# Axes
φ = collect(φnodes(latlon_grid, Center()))
z = collect(znodes(grid, Center()))

## Figure 11 -- Zonal-mean T, S, b

In [ ]:
fig = Figure(size = (1800, 500), fontsize = 14)

ax = Axis(fig[1, 1]; xlabel="Latitude", ylabel="Depth (m)", title="Zonal-mean temperature")
hm = heatmap!(ax, φ, z, T_zonal; colormap=:thermal, colorrange=(-2, 30), nan_color=:lightgray)
Colorbar(fig[1, 2], hm; label="deg C")
ylims!(ax, (-5500, 0))

ax = Axis(fig[1, 3]; xlabel="Latitude", ylabel="Depth (m)", title="Zonal-mean salinity")
hm = heatmap!(ax, φ, z, S_zonal; colormap=:haline, colorrange=(33, 37), nan_color=:lightgray)
Colorbar(fig[1, 4], hm; label="PSU")
ylims!(ax, (-5500, 0))

ax = Axis(fig[1, 5]; xlabel="Latitude", ylabel="Depth (m)", title="Zonal-mean buoyancy")
hm = heatmap!(ax, φ, z, b_zonal; colormap=:balance, nan_color=:lightgray)
Colorbar(fig[1, 6], hm; label="m/s^2")
ylims!(ax, (-5500, 0))

fig

## Figure 12 -- Zonal-mean drift from initial conditions

In [ ]:
fig = Figure(size = (1800, 500), fontsize = 14)

ax = Axis(fig[1, 1]; xlabel="Latitude", ylabel="Depth (m)", title="Zonal T - WOA")
hm = heatmap!(ax, φ, z, δT_zonal; colormap=:balance, colorrange=(-5, 5), nan_color=:lightgray)
Colorbar(fig[1, 2], hm; label="deg C")
ylims!(ax, (-5500, 0))

ax = Axis(fig[1, 3]; xlabel="Latitude", ylabel="Depth (m)", title="Zonal S - WOA")
hm = heatmap!(ax, φ, z, δS_zonal; colormap=:balance, colorrange=(-1, 1), nan_color=:lightgray)
Colorbar(fig[1, 4], hm; label="PSU")
ylims!(ax, (-5500, 0))

ax = Axis(fig[1, 5]; xlabel="Latitude", ylabel="Depth (m)", title="Zonal b - b(t=0)")
hm = heatmap!(ax, φ, z, δb_zonal; colormap=:balance, nan_color=:lightgray)
Colorbar(fig[1, 6], hm; label="m/s^2")
ylims!(ax, (-5500, 0))

fig